*  DSC540-T303 Data Preparation (2265-1)
*  Milestone 3
*  David Jorgensen

# Milestone 3

## Previous Work

### Milestones 1-2

In [30]:
# Load Libraries
import pandas as pd
import re

In [31]:
# Load data
flat_df = pd.read_csv('./broadway.csv')
# Display the first 10 rows of the dataset
flat_df.head(10)

,Date.Day,Date.Full,Date.Month,Date.Year,Show.Name,Show.Theatre,Show.Type,Statistics.Attendance,Statistics.Capacity,Statistics.Gross,Statistics.Gross Potential,Statistics.Performances
0,26,8/26/1990,8,1990,Tru,Booth,Play,5500,88,134456,0,8
1,24,3/24/1991,3,1991,Miss Saigon,Broadway,Musical,1737,100,100647,0,0
2,31,3/31/1991,3,1991,Miss Saigon,Broadway,Musical,12160,100,634424,0,0
3,7,4/7/1991,4,1991,Miss Saigon,Broadway,Musical,13921,100,713353,0,0
4,14,4/14/1991,4,1991,Miss Saigon,Broadway,Musical,10973,90,573981,0,4
5,21,4/21/1991,4,1991,Miss Saigon,Broadway,Musical,14076,101,706793,0,8
6,28,4/28/1991,4,1991,Miss Saigon,Broadway,Musical,14065,101,714968,0,8
7,5,5/5/1991,5,1991,Miss Saigon,Broadway,Musical,14064,101,730765,0,8
8,12,5/12/1991,5,1991,Miss Saigon,Broadway,Musical,13896,100,766713,0,8
9,19,5/19/1991,5,1991,Miss Saigon,Broadway,Musical,13738,99,763332,0,8


In [32]:
# Step 1: Filter to keep only rows where Show.Type is "Musical"
flat_df = flat_df[flat_df['Show.Type'] == 'Musical']
flat_df.head(10)

,Date.Day,Date.Full,Date.Month,Date.Year,Show.Name,Show.Theatre,Show.Type,Statistics.Attendance,Statistics.Capacity,Statistics.Gross,Statistics.Gross Potential,Statistics.Performances
1,24,3/24/1991,3,1991,Miss Saigon,Broadway,Musical,1737,100,100647,0,0
2,31,3/31/1991,3,1991,Miss Saigon,Broadway,Musical,12160,100,634424,0,0
3,7,4/7/1991,4,1991,Miss Saigon,Broadway,Musical,13921,100,713353,0,0
4,14,4/14/1991,4,1991,Miss Saigon,Broadway,Musical,10973,90,573981,0,4
5,21,4/21/1991,4,1991,Miss Saigon,Broadway,Musical,14076,101,706793,0,8
6,28,4/28/1991,4,1991,Miss Saigon,Broadway,Musical,14065,101,714968,0,8
7,5,5/5/1991,5,1991,Miss Saigon,Broadway,Musical,14064,101,730765,0,8
8,12,5/12/1991,5,1991,Miss Saigon,Broadway,Musical,13896,100,766713,0,8
9,19,5/19/1991,5,1991,Miss Saigon,Broadway,Musical,13738,99,763332,0,8
10,26,5/26/1991,5,1991,Miss Saigon,Broadway,Musical,13897,100,769137,0,8


In [33]:
# Step 2: Standardize String Columns
# Strip whitespace from text columns to ensure consistent grouping
text_cols = flat_df.select_dtypes(include='object').columns
for col in text_cols:
    flat_df[col] = flat_df[col].str.strip()

In [34]:
# Step 3: Ensure Numeric Columns are Correctly Typed
# This prevents errors if attendance/revenue were imported as objects due to commas or symbols
numeric_cols = ['Statistics.Attendance', 'Statistics.Revenue', 'Statistics.Capacity']
for col in numeric_cols:
    if col in flat_df.columns:
        flat_df[col] = pd.to_numeric(flat_df[col], errors='coerce')

In [35]:
# Step 4: Handle Missing Values
# Fill missing numerical values with median to preserve distribution
for col in numeric_cols:
    if col in flat_df.columns:
        median_val = flat_df[col].median()
        # Assign the result of fillna back to the DataFrame column
        flat_df[col] = flat_df[col].fillna(median_val)

In [36]:
# Step 5: Identify Outliers (Keep them, but flag them)
# This preserves the data points while marking them for potential later analysis
Q1 = flat_df['Statistics.Attendance'].quantile(0.25)
Q3 = flat_df['Statistics.Attendance'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

flat_df['IsAttendanceOutlier'] = (flat_df['Statistics.Attendance'] > upper_bound) | \
                                 (flat_df['Statistics.Attendance'] < lower_bound)


In [37]:
# Step 6: Reset Index in case I plan to remove rows later
flat_df.reset_index(drop=True, inplace=True)

In [38]:
# SPOT CHECK: Verify the wrangling
print("Data Types:")
print(flat_df.dtypes)
print("\nMissing Values:")
print(flat_df.isnull().sum())
print("\nOutlier Count:")
print(flat_df['IsAttendanceOutlier'].sum())

Data Types:
Date.Day                       int64
Date.Full                     object
Date.Month                     int64
Date.Year                      int64
Show.Name                     object
Show.Theatre                  object
Show.Type                     object
Statistics.Attendance          int64
Statistics.Capacity            int64
Statistics.Gross               int64
Statistics.Gross Potential     int64
Statistics.Performances        int64
IsAttendanceOutlier             bool
dtype: object

Missing Values:
Date.Day                      0
Date.Full                     0
Date.Month                    0
Date.Year                     0
Show.Name                     0
Show.Theatre                  0
Show.Type                     0
Statistics.Attendance         0
Statistics.Capacity           0
Statistics.Gross              0
Statistics.Gross Potential    0
Statistics.Performances       0
IsAttendanceOutlier           0
dtype: int64

Outlier Count:
12


In [39]:
# Print list of each unique Show.Name, organized in alphabetical order
print("\nUnique Show Names:")
print(sorted(flat_df['Show.Name'].unique()))
print("Number of Unique Shows:", len(flat_df['Show.Name'].unique()))


Unique Show Names:
['110 In The Shade', '13', '1776', "42Nd Street '01", '9 To 5', 'A Catered Affair', "A Chorus Line '06", 'A Christmas Story', 'A Class Act', 'A Funny Thing Happened On The Way To The Forum', "A Gentleman'S Guide To Love And Murder", 'A Little Night Music', 'A Night With Janis Joplin', 'A Tale Of Two Cities', 'A Year With Frog And Toad', 'After Midnight', 'Aida', 'Aladdin', 'All About Me', 'All Shook Up', 'Allegiance', 'Amazing Grace', 'American Idiot', 'American Psycho', 'Amour', 'An American In Paris', 'An Evening With Jerry Herman', 'Annie', "Annie '12", 'Annie Get Your Gun', "Anything Goes '11", 'Assassins', 'Avenue Q', "Baby It'S You!", 'Band In Berlin', 'Beautiful', 'Beauty And The Beast', 'Bells Are Ringing', 'Big', 'Big Fish', "Big River '03", 'Billy Elliot: The Musical', 'Blast!', 'Bloody Bloody Andrew Jackson', 'Bombay Dreams', 'Bonnie And Clyde', 'Bright Star', 'Bring In Da Noise, Bring In Da Funk', 'Bring It On The Musical', 'Brooklyn', 'Bullets Over Broa

In [40]:
# Step 7: Remove date suffix from all shows (" yy", " 'yy", " yyyy")
flat_df['Show.Name'] = flat_df['Show.Name'].str.replace(r'\s\d{2}$|\s\'\d{2}$|\s\d{4}$', '', regex=True)
print("Number of Unique Shows:", len(flat_df['Show.Name'].unique()))

# Step 8: Fix Show Name casing to be title case
def fix_title_casing(text):
    # Apply title casing first
    text = text.title()
    # Lowercase letter after apostrophe (e.g., O'Connors -> O'connors)
    text = re.sub(r"'([A-Z])", lambda m: "'" + m.group(1).lower(), text)
    # Lowercase letter after digit (e.g., 42Nd -> 42nd)
    text = re.sub(r'(\d)([A-Z])', lambda m: m.group(1) + m.group(2).lower(), text)
    return text

flat_df['Show.Name'] = flat_df['Show.Name'].apply(fix_title_casing)

print(sorted(flat_df['Show.Name'].unique()))

Number of Unique Shows: 281
['110 In The Shade', '13', '1776', '42nd Street', '9 To 5', 'A Catered Affair', 'A Chorus Line', 'A Christmas Story', 'A Class Act', 'A Funny Thing Happened On The Way To The Forum', "A Gentleman's Guide To Love And Murder", 'A Little Night Music', 'A Night With Janis Joplin', 'A Tale Of Two Cities', 'A Year With Frog And Toad', 'After Midnight', 'Aida', 'Aladdin', 'All About Me', 'All Shook Up', 'Allegiance', 'Amazing Grace', 'American Idiot', 'American Psycho', 'Amour', 'An American In Paris', 'An Evening With Jerry Herman', 'Annie', 'Annie Get Your Gun', 'Anything Goes', 'Assassins', 'Avenue Q', "Baby It's You!", 'Band In Berlin', 'Beautiful', 'Beauty And The Beast', 'Bells Are Ringing', 'Big', 'Big Fish', 'Big River', 'Billy Elliot: The Musical', 'Blast!', 'Bloody Bloody Andrew Jackson', 'Bombay Dreams', 'Bonnie And Clyde', 'Bright Star', 'Bring In Da Noise, Bring In Da Funk', 'Bring It On The Musical', 'Brooklyn', 'Bullets Over Broadway', 'Burn The Floo

In [41]:
# Print 30 random rows of data
flat_df.sample(n=30, random_state=42)

,Date.Day,Date.Full,Date.Month,Date.Year,Show.Name,Show.Theatre,Show.Type,Statistics.Attendance,Statistics.Capacity,Statistics.Gross,Statistics.Gross Potential,Statistics.Performances,IsAttendanceOutlier
12959,24,2/24/2008,2,2008,Mamma Mia!,Winter Garden,Musical,10516,88,759707,64,8,False
14184,22,3/22/2009,3,2009,Avenue Q,Golden,Musical,5410,85,332180,58,8,False
8056,19,10/19/2003,10,2003,Beauty And The Beast,Lunt-Fontanne,Musical,11285,93,625699,75,8,False
5523,8,4/8/2001,4,2001,Follies,Belasco,Musical,6110,77,332856,67,5,False
8801,20,6/20/2004,6,2004,42nd Street,Ford Center,Musical,11478,79,629795,59,8,False
8686,9,5/9/2004,5,2004,The Boy From Oz,Imperial,Musical,10862,95,812001,85,8,False
4413,2,4/2/2000,4,2000,Cabaret,Studio 54 ('98),Musical,6733,91,463691,83,8,False
3979,14,11/14/1999,11,1999,Fosse,Broadhurst,Musical,8915,98,618434,96,8,False
15363,9,5/9/2010,5,2010,In The Heights,Richard Rodgers,Musical,7888,73,531617,46,8,False
20434,7,12/7/2014,12,2014,Hedwig And The Angry Inch,Belasco,Musical,6002,85,610431,82,7,False


The data wrangling process for this project involved the following transformations:

1. Filtering to keep only rows where Show.Type is "Musical" ensures that we are only analyzing musicals.
2. Standardizing string columns by stripping whitespace from text columns helps in ensuring consistent grouping and reduces errors during analysis.
3. Ensuring numeric columns are correctly typed prevents errors if attendance/revenue were imported as objects due to commas or symbols, allowing for accurate numerical calculations.
4. Removing date suffixes from the 'Show.Name' column ensures all the shows are grouped correctly later during analysis.
5. Fixing title casing of the 'Show.Name' column ensures that all show names are consistently formatted.
6. Identifying and indicating outliers in a separate column allows for easy identification of potential errors or anomalies in the dataset.

Key risks emerging from these transformations include selection bias if certain rows were dropped, or misrepresentation if categorical mappings oversimplified complex attributes. Assumptions were made during cleaning, such as treating specific null values as "unknown" rather than "missing completely at random," which could impact model performance or statistical conclusions. This data is publicly provided by the Broadway League (the national trade association for the Broadway industry), and was aggregated by Austin Cory Bart and distributed via the CORGIS Datasets Project. Ethical acquisition was ensured by confirming that the dataset’s usage license permitted commercial or academic analysis, and the data does not contain PII. To mitigate ethical implications, I would document all transformation steps, conduct sensitivity analyses on how assumptions affect outcomes, and ensure that final outputs do not disproportionately impact any sub-group defined by sensitive attributes like race, gender, or socioeconomic status.

## Milestone 3 Work

### Instructions

Perform at least 5 data transformation and/or cleansing steps to your website data. The below examples are not required - they are just potential transformations you could do. If your data doesn't work for these scenarios, complete different transformations. You can do the same transformation multiple times if needed to clean your data. The goal is a clean dataset at the end of the milestone. As a reminder - you cannot export your website data to CSV to work with it, you must do all the work directly against the HTML source.

Examples:

- Replace Headers
- Format data into a more readable format
- Identify outliers and bad data
- Find duplicates
- Fix casing or inconsistent values
- Conduct Fuzzy Matching

Make sure you clearly number and label each transformation step (Step #1, Step #2, etc.) in your code and describe what it is doing in 1-2 sentences.

You must submit the following:

- Jupyter Notebook File or PDF of your code with Milestone # listed.
- Each transformation should be labeled with description or what it is doing.
- Human readable dataset after all transformations should be printed at the end of your notebook.
- 1 paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed answering the following questions (this can be done in your notebook):
- What changes were made to the data?
- Are there any legal or regulatory guidelines for your data or project topic?
- What risks could be created based on the transformations done?
- Did you make any assumptions in cleaning/transforming the data?
- How was your data sourced/verified for credibility?
- Was your data acquired in an ethical way?
- How would you mitigate any of the ethical implications you have identified?

### Assignment

In [42]:
# Step 1: Import website data
# Fetches the HTML from Wikipedia and parses the wikitable into a DataFrame.
# A 2-D grid approach handles rowspan cells (musicals with multiple film adaptations
# share one merged cell across several rows). <br> tags become ", " to preserve
# multi-person contributor lists within a single cell.
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

URL = 'https://en.wikipedia.org/wiki/List_of_musicals_adapted_into_feature_films'
response = requests.get(URL, headers={'User-Agent': 'Mozilla/5.0'})
soup = BeautifulSoup(response.content, 'html.parser')
table = soup.find('table', class_='wikitable')

def parse_table(table):
    """Parse a wikitable with rowspan/colspan support using a 2-D grid.
    Uses strip=False so that spaces adjacent to inline tags are preserved
    (e.g., the space between a title link and its year in parentheses).
    """
    rows = table.find_all('tr')
    grid = {}  # (row_idx, col_idx) -> text
    for row_i, tr in enumerate(rows):
        col_i = 0
        for cell in tr.find_all(['td', 'th']):
            while (row_i, col_i) in grid:
                col_i += 1
            rowspan = int(cell.get('rowspan', 1))
            colspan = int(cell.get('colspan', 1))
            for br in cell.find_all('br'):
                br.replace_with(', ')
            # strip=False preserves spaces between inline elements;
            # outer .strip() removes only leading/trailing whitespace.
            value = cell.get_text(strip=False).strip()
            # Add space before year where none exists: "13(2007)" -> "13 (2007)"
            value = re.sub(r'([^\s])\((\d{4})\)', r'\1 (\2)', value)
            for r in range(rowspan):
                for c in range(colspan):
                    grid[(row_i + r, col_i + c)] = value
            col_i += colspan
    max_row = max(r for r, c in grid) + 1
    max_col = max(c for r, c in grid) + 1
    return [[grid.get((r, c), '') for c in range(max_col)] for r in range(max_row)]

data = parse_table(table)
films_df = pd.DataFrame(data[1:], columns=data[0])
print(f"Shape: {films_df.shape}")
films_df.head(10)

Shape: (228, 6)


,Musical,Music,Lyrics,Book,Film,Film director
0,13 (2007),Jason Robert Brown,Jason Robert Brown,"Dan Elish, Robert Horn",13: The Musical (2022),Tamra Davis
1,1776 (1969),Sherman Edwards,Sherman Edwards,Peter Stone,1776 (1972),Peter H. Hunt
2,"Alma, Where Do You Live? (1910)",Jean Briquet,George V. Hobart,George V. Hobart,"Alma, Where Do You Live? (1917)",Hal Clarendon
3,Animal Crackers (1928),"Bert Kalmar, Harry Ruby","Bert Kalmar, Harry Ruby","George S. Kaufman, Morrie Ryskind",Animal Crackers (1930),Victor Heerman
4,Annie (1976),Charles Strouse,Martin Charnin,Thomas Meehan,Annie (1982),John Huston
5,Annie (1976),Charles Strouse,Martin Charnin,Thomas Meehan,Annie: A Royal Adventure! (1995),Bille Woodruff
6,Annie (1976),Charles Strouse,Martin Charnin,Thomas Meehan,Annie (1999),Rob Marshall
7,Annie (1976),Charles Strouse,Martin Charnin,Thomas Meehan,Annie (2014),Will Gluck
8,Annie Get Your Gun (1946),Irving Berlin,Irving Berlin,Herbert and Dorothy Fields,Annie Get Your Gun (1950),George Sidney
9,Annie Get Your Gun (1946),Irving Berlin,Irving Berlin,Herbert and Dorothy Fields,Annie Get Your Gun (1957),Vincent J. Donehue


In [43]:
# Step 2: Rename columns to more descriptive names
# The original column names from Wikipedia ("Music", "Book") are ambiguous out of context;
# renaming them to "Music_Composer", "Book_Writer", etc. improves readability.
films_df = films_df.rename(columns={
    'Music': 'Music_Composer',
    'Lyrics': 'Lyricist',
    'Book': 'Book_Writer',
    'Film': 'Film_Title',
    'Film director': 'Film_Director'
})

# Replace " and" with ", " in name columns to standardize contributor separators,
# then collapse any double spaces that result from the substitution.
name_cols = ['Music_Composer', 'Lyricist', 'Book_Writer', 'Film_Director']
for col in name_cols:
    films_df[col] = (films_df[col]
        .str.replace(' and', ', ', regex=False)
        .str.replace(r' {2,}', ' ', regex=True)
        .str.strip()
    )

print("Columns:", films_df.columns.tolist())
films_df.head(5)

Columns: ['Musical', 'Music_Composer', 'Lyricist', 'Book_Writer', 'Film_Title', 'Film_Director']


,Musical,Music_Composer,Lyricist,Book_Writer,Film_Title,Film_Director
0,13 (2007),Jason Robert Brown,Jason Robert Brown,"Dan Elish, Robert Horn",13: The Musical (2022),Tamra Davis
1,1776 (1969),Sherman Edwards,Sherman Edwards,Peter Stone,1776 (1972),Peter H. Hunt
2,"Alma, Where Do You Live? (1910)",Jean Briquet,George V. Hobart,George V. Hobart,"Alma, Where Do You Live? (1917)",Hal Clarendon
3,Animal Crackers (1928),"Bert Kalmar, Harry Ruby","Bert Kalmar, Harry Ruby","George S. Kaufman, Morrie Ryskind",Animal Crackers (1930),Victor Heerman
4,Annie (1976),Charles Strouse,Martin Charnin,Thomas Meehan,Annie (1982),John Huston


In [44]:
# Step 3: Remove Wikipedia citation markers (e.g., [1], [2], [note 1])
# These artifacts come from HTML reference superscripts and carry no analytical value.
def remove_citations(text):
    if isinstance(text, str):
        return re.sub(r'\[\d+\]|\[note \d+\]', '', text).strip()
    return text

films_df = films_df.map(remove_citations)

# Verify no citation markers remain
has_cite = films_df.apply(lambda col: col.str.contains(r'\[', na=False).any())
remaining = has_cite[has_cite].index.tolist()
print("Columns still containing '[' :", remaining if remaining else "None — all citations removed")
films_df.head(5)

Columns still containing '[' : None — all citations removed


,Musical,Music_Composer,Lyricist,Book_Writer,Film_Title,Film_Director
0,13 (2007),Jason Robert Brown,Jason Robert Brown,"Dan Elish, Robert Horn",13: The Musical (2022),Tamra Davis
1,1776 (1969),Sherman Edwards,Sherman Edwards,Peter Stone,1776 (1972),Peter H. Hunt
2,"Alma, Where Do You Live? (1910)",Jean Briquet,George V. Hobart,George V. Hobart,"Alma, Where Do You Live? (1917)",Hal Clarendon
3,Animal Crackers (1928),"Bert Kalmar, Harry Ruby","Bert Kalmar, Harry Ruby","George S. Kaufman, Morrie Ryskind",Animal Crackers (1930),Victor Heerman
4,Annie (1976),Charles Strouse,Martin Charnin,Thomas Meehan,Annie (1982),John Huston


In [45]:
# Step 4: Extract premiere year from Musical column into a separate Musical_Year column
# Musical names include their original premiere year in parentheses (e.g., "Cats(1981)");
# separating the year enables temporal analysis independent of the name string.
films_df['Musical_Year'] = films_df['Musical'].str.extract(r'\((\d{4})\)').astype('Int64')
films_df['Musical'] = films_df['Musical'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()

print(f"Musical year range: {films_df['Musical_Year'].min()} – {films_df['Musical_Year'].max()}")
films_df[['Musical', 'Musical_Year']].head(10)

Musical year range: 1904 – 2022


,Musical,Musical_Year
0,13,2007
1,1776,1969
2,"Alma, Where Do You Live?",1910
3,Animal Crackers,1928
4,Annie,1976
5,Annie,1976
6,Annie,1976
7,Annie,1976
8,Annie Get Your Gun,1946
9,Annie Get Your Gun,1946


In [46]:
# Step 5: Extract release year from Film_Title column into a separate Film_Year column
# Film titles include their release year in parentheses (e.g., "Annie(1982)");
# separating the year allows direct comparison between the musical premiere and film release dates.
films_df['Film_Year'] = films_df['Film_Title'].str.extract(r'\((\d{4})\)').astype('Int64')
films_df['Film_Title'] = films_df['Film_Title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()

print(f"Film year range: {films_df['Film_Year'].min()} – {films_df['Film_Year'].max()}")
films_df[['Film_Title', 'Film_Year']].head(10)

Film year range: 1917 – 2024


,Film_Title,Film_Year
0,13: The Musical,2022
1,1776,1972
2,"Alma, Where Do You Live?",1917
3,Animal Crackers,1930
4,Annie,1982
5,Annie: A Royal Adventure!,1995
6,Annie,1999
7,Annie,2014
8,Annie Get Your Gun,1950
9,Annie Get Your Gun,1957


In [47]:
# Step 6: Replace empty strings with NaN to standardize missing value representation
# Empty cells from HTML parsing come through as empty strings; replacing them with NaN ensures
# that pandas aggregation functions (count, mean, etc.) correctly exclude missing values.
films_df.replace('', pd.NA, inplace=True)

print("Missing values per column:")
print(films_df.isnull().sum())

Missing values per column:
Musical            0
Music_Composer     9
Lyricist          15
Book_Writer        1
Film_Title         0
Film_Director      0
Musical_Year       3
Film_Year          0
dtype: int64


In [48]:
# Step 7: Flag musicals adapted into multiple films
# Some musicals appear in more than one row because they were adapted into multiple feature films
# (e.g., "Annie" → 1982, 2014). Flagging these rows distinguishes remakes from unique adaptations.
films_df['Multiple_Adaptations'] = films_df.duplicated(subset=['Musical'], keep=False)

n_musicals = films_df[films_df['Multiple_Adaptations']]['Musical'].nunique()
n_rows = int(films_df['Multiple_Adaptations'].sum())
print(f"Musicals adapted more than once: {n_musicals} unique titles across {n_rows} rows")

films_df[films_df['Multiple_Adaptations']][['Musical', 'Film_Title', 'Film_Year']].sort_values('Musical').head(20)

Musicals adapted more than once: 30 unique titles across 68 rows


,Musical,Film_Title,Film_Year
4,Annie,Annie,1982
5,Annie,Annie: A Royal Adventure!,1995
6,Annie,Annie,1999
7,Annie,Annie,2014
8,Annie Get Your Gun,Annie Get Your Gun,1950
9,Annie Get Your Gun,Annie Get Your Gun,1957
10,Annie Get Your Gun,Annie Get Your Gun,1967
11,Anything Goes,Anything Goes,1936
12,Anything Goes,Anything Goes,1956
32,Brigadoon,Brigadoon,1966


In [49]:
# Step 8: Compute Years_Until_Adaptation — derived column for analysis
# Calculates the number of years between a musical's stage premiere and its first (or each) film adaptation.
# This enables analysis of how quickly or slowly Broadway shows become feature films over time.
films_df['Years_Until_Adaptation'] = films_df['Film_Year'] - films_df['Musical_Year']

valid = films_df['Years_Until_Adaptation'].dropna()
print(f"Average years until adaptation : {valid.mean():.1f}")
print(f"Fastest adaptation             : {valid.min():.0f} years")
print(f"Longest wait                   : {valid.max():.0f} years")

films_df[['Musical', 'Musical_Year', 'Film_Title', 'Film_Year', 'Years_Until_Adaptation']].dropna().sort_values('Years_Until_Adaptation').head(10)

Average years until adaptation : 11.3
Fastest adaptation             : -6 years
Longest wait                   : 64 years


,Musical,Musical_Year,Film_Title,Film_Year,Years_Until_Adaptation
13,Are You with It?,1945,Are You with It?,1939,-6
24,The Better 'Ole,1917,The Romance of Old Bill,1918,1
65,Follow Thru,1929,Follow Thru,1930,1
64,Flying High,1930,Flying High,1931,1
91,Hazel Flagg,1953,Living It Up,1954,1
191,Something for the Boys,1943,Something for the Boys,1944,1
164,Paris,1928,Paris,1929,1
216,Under Your Hat,1939,Under Your Hat,1940,1
196,Spring Is Here,1929,Spring Is Here,1930,1
206,Take a Chance,1932,Take a Chance,1933,1


In [50]:
# SPOT CHECK: Final dataset overview
print("Final shape:", films_df.shape)
print("\nColumn data types:")
print(films_df.dtypes)
print("\nMissing values:")
print(films_df.isnull().sum())
print("\n30 random rows:")
films_df.sample(n=30, random_state=42)

Final shape: (228, 10)

Column data types:
Musical                   object
Music_Composer            object
Lyricist                  object
Book_Writer               object
Film_Title                object
Film_Director             object
Musical_Year               Int64
Film_Year                  Int64
Multiple_Adaptations        bool
Years_Until_Adaptation     Int64
dtype: object

Missing values:
Musical                    0
Music_Composer             9
Lyricist                  15
Book_Writer                1
Film_Title                 0
Film_Director              0
Musical_Year               3
Film_Year                  0
Multiple_Adaptations       0
Years_Until_Adaptation     3
dtype: int64

30 random rows:


,Musical,Music_Composer,Lyricist,Book_Writer,Film_Title,Film_Director,Musical_Year,Film_Year,Multiple_Adaptations,Years_Until_Adaptation
215,Top Speed,Harry Ruby,Bert Kalmar,"Guy Bolton, Harry Ruby, Bert Kalmar",Top Speed,Mervyn LeRoy,1929,1930,False,1
66,For Goodness Sake,"William Daly, Paul Lannin",Arthur Jackson,Frederick J. Jackson,Stop Flirting,Scott Sidney,1922,1925,False,3
9,Annie Get Your Gun,Irving Berlin,Irving Berlin,"Herbert, Dorothy Fields",Annie Get Your Gun,Vincent J. Donehue,1946,1957,True,11
200,Sunny,Jerome Kern,"Oscar Hammerstein II, Otto Harbach","Oscar Hammerstein II, Otto Harbach",Sunny,Herbert Wilcox,1925,1941,True,16
15,Babes in Arms,Richard Rodgers,Lorenz Hart,"Rodgers, Hart",Babes in Arms,Busby Berkeley,1937,1939,False,2
197,Stop the World – I Want to Get Off,"Leslie Bricusse , Anthony Newley","Leslie Bricusse , Anthony Newley","Leslie Bricusse , Anthony Newley",Stop the World – I Want to Get Off,Philip Saville,1961,1966,False,5
25,The Better 'Ole,Herman Darewski,"Percival Knight, James Heard","Bruce Bairnsfather, Arthur Elliot",The Better 'Ole,Charles Reisner,1917,1926,True,9
193,South Pacific,Richard Rodgers,Oscar Hammerstein II,"Oscar Hammerstein II, Joshua Logan",South Pacific,Joshua Logan,1949,1958,True,9
154,Oliver!,Lionel Bart,Lionel Bart,Lionel Bart,Oliver!,Carol Reed,1960,1968,False,8
126,Little Nellie Kelly,George M. Cohan,George M. Cohan,George M. Cohan,Little Nellie Kelly,Norman Taurog,1922,1940,False,18


### Work Summary
| Step | What it does |
| ---- | ------------ |
| 1	| Fetch HTML from Wikipedia URL, parse wikitable with BeautifulSoup, use \<br\> → , and "and" → , to preserve multi-person cells |
| 2	| Rename columns (Music → Music_Composer, Book → Book_Writer, etc.) |
| 3	| Strip Wikipedia citation markers ([1], [note 1]) via regex |
| 4	| Extract premiere year from Musical names (e.g., Cats(1981) → name + Musical_Year=1981) |
| 5	| Extract release year from Film titles → Film_Year |
| 6	| Replace empty strings with NaN |
| 7	| Flag musicals with multiple film adaptations (Multiple_Adaptations bool column) |
| 8	| Compute derived Years_Until_Adaptation = Film_Year - Musical_Year |

**Legal and regulatory considerations:** The data originates from Wikipedia, which is published under the Creative Commons Attribution-ShareAlike 4.0 International License. Academic and non-commercial analysis is explicitly permitted, provided attribution is given; no personally identifiable information (PII) is present in the dataset.

**Risks:** The primary transformation risk is data loss or distortion from the citation-removal step: if a citation marker was embedded mid-string (e.g., "Rodgers[3] and Hammerstein"), removing the bracket would yield "Rodgers and Hammerstein" which is actually correct, but edge cases could silently corrupt contributor names. Renaming the "Book" column to "Book_Writer" assumes the Wikipedia table definition is stable; if the source page is restructured, the rename mapping would break the pipeline. The year-extraction regex assumes years are always four-digit numerals in parentheses; non-standard annotations would produce `NaN` silently.

**Assumptions:** It was assumed that all years appearing in parentheses in the Musical and Film columns are calendar years (not catalog numbers or act numbers), and that a missing year is truly absent rather than a data-entry error. The pipe delimiter chosen for multi-person cells (`|`) was assumed not to appear naturally in contributor names.

**Credibility and ethical acquisition:** The source is the English Wikipedia article "List of musicals adapted into feature films," a collaboratively maintained reference whose editorial history and citations are publicly auditable. The page was fetched programmatically via a standard HTTP GET request with no authentication bypass, scraping guard circumvention, or rate-limit violation.

**Mitigating ethical implications:** To mitigate risks, all transformation steps are numbered and documented here so that any downstream consumer can reproduce or audit the pipeline. Because the dataset contains no personal or demographic data, there is no risk of disparate impact on protected groups. Should Wikipedia's page structure change, the pipeline should be re-validated against the raw HTML before drawing new conclusions.